# Wordle: GPT-2 + LoRA + ES

Train a **linear head** on a Hugging Face causal LM with **Evolution Strategies**. The hyperparameter cell defaults to **`TRAIN_BUDGET="long"`** (more iterations, larger `N_POP`, longer warm-start). Set **`TRAIN_BUDGET="fast"`** for a short smoke test. Use **GPU** for long runs; CPU will take a long time.

**Also:** Richer prompts with **structured constraints** from feedback; optional **supervised warm-start** (cross-entropy to the secret word; label only, not in the prompt).

**Requires:** `torch`, `transformers`, `numpy`, `matplotlib`. **`pip install peft`** only if `USE_LORA=True` in §2. First run downloads HF weights.

## 1. Environment and imports

`peft` is **not** imported here (only needed for `USE_LORA=True`). Install it yourself if you enable LoRA.

**If you see `unexpected keyword argument 'normalize_gradient'`:** restart the kernel and run this cell again, or rely on `importlib.reload` below (re-run this cell after editing `src/es_wordle.py`).

In [ ]:
import sys
import importlib
from pathlib import Path

import numpy as np
import torch

# Repo root + src on path (works from `notebooks/` or project root)
_here = Path.cwd().resolve()
ROOT = _here.parent if _here.name == "notebooks" else _here
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from wordle_env import load_wordle_environment
from wordle_gpt2_policy import WordleGPT2Policy, TRANSFORMERS_AVAILABLE

# Always pick up the latest es_wordle.py (Jupyter caches imports)
import es_wordle
importlib.reload(es_wordle)
from es_wordle import train_es_wordle

import inspect
_sig = inspect.signature(train_es_wordle)
for _name in ("normalize_gradient", "eval_deterministic"):
    if _name not in _sig.parameters:
        raise RuntimeError(
            f"src/es_wordle.py is missing {_name}. Re-run the import cell after `importlib.reload`, or pull the latest repo."
        )

if not TRANSFORMERS_AVAILABLE:
    raise ImportError("Install transformers: pip install transformers")

print("ROOT:", ROOT)
print("es_wordle:", es_wordle.__file__)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 2. Hyperparameters

**`TRAIN_BUDGET`** in the next cell: **`"long"`** (default) runs more ES iterations, larger population, longer warm-start, and **2 rollouts per fitness member** — still slow on CPU; use **GPU** if you can. Set **`"fast"`** for a quick smoke test.

- **`MODEL_NAME`**: **`distilgpt2`** is much faster on CPU than **`gpt2`**.
- **`USE_LORA`**: **`False`** trains only the head (fastest ES). **`True`** adapts the backbone (needs `peft`; many more trainable params ⇒ slower).
- **`MOCK_ENV`**: `True` avoids the Prime Intellect `verifiers` dependency.
- **`MAX_VOCAB`**: smaller ⇒ **fewer head weights**, **faster ES**, easier exploration. **`long`** uses **64** words; **`fast`** uses **256**. Mock targets are **listed first** — need `MAX_VOCAB ≥ 8`. Narrow vocab is bad if you later need full English Wordle.
- **`N_POP` / `N_ITERATIONS`**: main **wall-clock** levers for ES (each iteration runs `N_POP` full rollouts).
- **`N_EVAL_EPISODES`**: episodes **per fitness member** (inner ES loop). **`long`** budget uses **2** for less noisy fitness (2× cost per iter).
- **`EVAL_EVERY` / `EVAL_N_EPISODES`**: periodic **logging** eval only (e.g. iters **0, 10, 20, …** with `EVAL_EVERY=10`). Large `EVAL_N_EPISODES` slows those steps.
- **`NORMALIZE_GRADIENT`**: `True` uses `θ ← θ + α·ĝ/‖ĝ‖` when ‖ĝ‖ is huge.
- **`RANK_FITNESS`**: rank-normalized fitness (often better when `N_POP` is small).
- **`EVAL_DETERMINISTIC`**: `False` matches stochastic ES rollouts for the Success % column.
- **`FITNESS_OBJECTIVE`**: default **`win_plus_return`** makes ES optimize **`WIN_FITNESS_SCALE * win_rate + mean_return`** so wins matter more than partial-credit return alone. Use **`return`** to restore the old behavior.
- **`RICHER_PROMPT`**: constraint summary from `wordle_hints.py`.
- **`WARM_START_STEPS`**: supervised steps before ES (**0** = skip; lower = faster).

In [ ]:
# --- Training budget: "long" = actually train more (slow on CPU; GPU strongly recommended) ---
TRAIN_BUDGET = "long"  # "fast" | "long"

SEED = 42
MODEL_NAME = "distilgpt2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MOCK_ENV = True
USE_PRIME_TARGETS = False

SIGMA = 0.03
ALPHA = 0.15
NORMALIZE_GRADIENT = True
RANK_FITNESS = True
EVAL_DETERMINISTIC = False

FITNESS_OBJECTIVE = "win_plus_return"
WIN_FITNESS_SCALE = 5.0

USE_LORA = False  # True on GPU + peft for more capacity (much slower)
LORA_R = 4
RICHER_PROMPT = True
WARM_START_LR = 3e-4

if TRAIN_BUDGET == "fast":
    MAX_VOCAB = 256
    N_EVAL_EPISODES = 1
    EVAL_N_EPISODES = 8
    EVAL_EVERY = 10
    WARM_START_STEPS = 120
    N_POP = 8
    N_ITERATIONS = 40
else:
    # Tighter action space: smaller linear head, faster ES, easier to "stumble" on wins
    MAX_VOCAB = 64
    N_EVAL_EPISODES = 2
    EVAL_N_EPISODES = 12
    EVAL_EVERY = 10
    WARM_START_STEPS = 300
    N_POP = 16
    N_ITERATIONS = 100

np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(DEVICE)
print(
    f"TRAIN_BUDGET={TRAIN_BUDGET} | Model: {MODEL_NAME} | LoRA: {USE_LORA} | device: {device}\n"
    f"  MAX_VOCAB={MAX_VOCAB}  ES: N_POP={N_POP}  N_ITERATIONS={N_ITERATIONS}  "
    f"n_eval_episodes={N_EVAL_EPISODES}  WARM_START_STEPS={WARM_START_STEPS}"
)

## 3. Build policy and environment

First execution downloads the tokenizer + weights for `MODEL_NAME` into the HF cache.

With `include_mock_targets_in_vocab=True` (default), the first actions include all `MOCK_WORDLE_TARGETS` so a random mock secret word is always in the action set.

Set `USE_LORA=True` in §2 for PEFT LoRA — run **`pip install peft`** before this cell. With `USE_LORA=False`, only the head is trained (faster ES).

In [ ]:
policy = WordleGPT2Policy(
    model_name=MODEL_NAME,
    use_prime_targets=USE_PRIME_TARGETS,
    max_vocab_size=MAX_VOCAB,
    include_mock_targets_in_vocab=True,
    richer_prompt=RICHER_PROMPT,
    use_lora=USE_LORA,
    lora_r=LORA_R,
).to(device)

from wordle_env import MOCK_WORDLE_TARGETS
assert all(w in policy.words for w in MOCK_WORDLE_TARGETS)

print(f"Trainable (ES): {policy.count_trainable_parameters():,}")
print(f"Total params:   {policy.count_parameters():,}")
print(f"Action dim:     {policy.action_dim}")

env = load_wordle_environment(
    num_train_examples=2000,
    num_eval_examples=20,
    use_prime_intellect=not MOCK_ENV,
)

## 4. Optional supervised warm-start

Random play for 1–4 guesses, then **cross-entropy** toward the secret word (the answer is **not** in the prompt). Set `WARM_START_STEPS = 0` to skip.

In [ ]:
from wordle_gpt2_warmstart import supervised_warm_start_wordle

if WARM_START_STEPS > 0:
    ws = supervised_warm_start_wordle(
        policy,
        env,
        n_steps=WARM_START_STEPS,
        lr=WARM_START_LR,
        device=device,
        seed=SEED,
        verbose=True,
    )
    print("Warm-start: fitted", len(ws["loss"]), "steps; skipped", ws["skipped"])
else:
    print("Skipping warm-start.")


## 5. Run ES training

`train_es_wordle` prints **every iteration** when `verbose=True`. **`‖θ−θ₀‖`** is the L2 distance of trainable weights from their **initial** values — it should **grow** every step (proof the optimizer is changing parameters). **`Step‖`** is often **constant** when `NORMALIZE_GRADIENT=True` (fixed step length by design). **`popσ`** is spread of fitness across the ES population (noise).

`history` now includes **`train_iter`, `train_fitness`, `param_drift`, `pop_fitness_std`, …** every iteration (not only eval checkpoints). The plot cell uses them in the **bottom row**.

Full eval lines (Success / Eval Reward) appear when `iteration % EVAL_EVERY == 0` or on the last iteration.

In [ ]:
history = train_es_wordle(
    policy=policy,
    env=env,
    N=N_POP,
    sigma=SIGMA,
    alpha=ALPHA,
    n_iterations=N_ITERATIONS,
    n_eval_episodes=N_EVAL_EPISODES,
    max_turns=6,
    eval_every=EVAL_EVERY,
    verbose=True,
    normalize_gradient=NORMALIZE_GRADIENT,
    eval_n_episodes=EVAL_N_EPISODES,
    rank_fitness=RANK_FITNESS,
    eval_deterministic=EVAL_DETERMINISTIC,
    fitness_objective=FITNESS_OBJECTIVE,
    win_fitness_scale=WIN_FITNESS_SCALE,
)

## 6. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

it = history["iteration"]
fig, axes = plt.subplots(2, 3, figsize=(12, 6))

axes[0, 0].plot(it, history["eval_reward"], "b-o", ms=3)
axes[0, 0].set_title("Eval mean reward")
axes[0, 0].set_xlabel("iteration")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(it, history["eval_success"], "g-o", ms=3)
axes[0, 1].set_title("Eval success rate")
axes[0, 1].set_xlabel("iteration")
axes[0, 1].set_ylim(0, 1.05)
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(it, history["eval_turns"], "m-o", ms=3)
axes[0, 2].set_title("Mean turns (eval)")
axes[0, 2].set_xlabel("iteration")
axes[0, 2].grid(True, alpha=0.3)

ti = history["train_iter"]
axes[1, 0].plot(ti, history["param_drift"], "k-", lw=1)
axes[1, 0].set_title("‖θ − θ₀‖ (params are updating)")
axes[1, 0].set_xlabel("iteration")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(ti, history["train_fitness"], "c-", lw=1, alpha=0.8)
axes[1, 1].set_title("ES mean fitness (every iter)")
axes[1, 1].set_xlabel("iteration")
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(ti, history["pop_fitness_std"], "orange", lw=1)
axes[1, 2].set_title("Population fitness σ (spread)")
axes[1, 2].set_xlabel("iteration")
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle(f"Wordle ES | {MODEL_NAME} + head")
plt.tight_layout()
plt.show()

## 7. Save checkpoint (optional)

Saves the **head** (and **LoRA adapter** if `USE_LORA=True`) plus `words` and `history` under `models/` at the repo root.

In [ ]:
save_path = ROOT / "models" / "wordle_gpt2_es_head.ipynb_run.pt"
save_path.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "head": policy.head.state_dict(),
    "model_name": MODEL_NAME,
    "words": policy.words,
    "history": history,
    "use_lora": USE_LORA,
}
if getattr(policy, "_lm_trainable", False):
    payload["lm"] = policy.lm.state_dict()
torch.save(payload, save_path)
print("Saved:", save_path)